# 20 · Results (spec §9)
All numbers here are computed by `purva_netra.pipeline` on **held-out years only** (primary: train 2018–2020, calibration 2021, test 2022; plus leave-one-year-out). Brier/BSS from `scores` and scikit-learn (cross-checked), reliability from `xskillscore`, CIs by whole-week block bootstrap. If the archive is incomplete this notebook says so and reports nothing.

In [1]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
P = Path('../data/processed/eval/results.json')
HAVE = P.exists()
print('results available:', HAVE)
if not HAVE:
    print('NOT RUN: the 2018–2022 archive has not been extracted yet (see README → Status). No skill numbers are reported.')

results available: False
NOT RUN: the 2018–2022 archive has not been extracted yet (see README → Status). No skill numbers are reported.


In [2]:
if HAVE:
    R = json.load(open(P))
    print('shipped:', R['shipped'], '| gate passed:', R['gate_passed'], '| version:', R['version'])
    print('features:', R['features'])
    display(pd.DataFrame(R['base_rate']).round(3))

## Primary split (test 2022): BSS vs B0 and B2 with 95% week-block CIs, ROC/PR-AUC, ECE, hit rate @ 20% FAR

In [3]:
if HAVE:
    display(pd.DataFrame(R['primary']).round(4))

## Skill by lead (test 2022)

In [4]:
if HAVE:
    bl = pd.DataFrame(R['primary_by_lead'])
    display(bl.round(3))
    fig, ax = plt.subplots(figsize=(8,4))
    for m, c in [('b2', '#eb6834'), ('m', '#1baf7a')]:
        s = bl[bl.model == m]
        ax.errorbar(s.lead, s.bss_b0, yerr=[s.bss_b0 - s.lo_b0, s.hi_b0 - s.bss_b0], color=c, lw=2, capsize=3, label={'b2':'B2 spread','m':'model'}[m])
    ax.axhline(0, color='#2a78d6', lw=1, label='B0 climatology'); ax.set_xlabel('lead day'); ax.set_ylabel('BSS vs B0'); ax.legend(frameon=False)

## Leave-one-year-out and the §9 gate (BSS vs B2, lower 95% bound > 0 for Day 3–10)

In [5]:
if HAVE and R['loyo_by_lead']:
    lb = pd.DataFrame(R['loyo_by_lead'])
    g = lb[(lb.model == 'm') & lb.lead.between(3, 10)][['lead','bss_b2','lo_b2','hi_b2']]
    display(g.round(4)); print('GATE PASSED' if (g.lo_b2 > 0).all() and len(g) == 8 else 'GATE NOT PASSED — ship B2 + analogs + calibration')
    display(pd.DataFrame(R['loyo']).round(4))

## Reliability (test 2022, 10 bins) — xskillscore vs scikit-learn cross-check

In [6]:
if HAVE:
    from purva_netra.evaluate import library_report
    pr = pd.read_parquet('../data/processed/eval/primary_predictions.parquet')
    te = pr[(pr.init.dt.year == 2022) & (pr.bust >= 0) & pr.p_b2.notna()]
    fig, ax = plt.subplots(figsize=(5,5)); ax.plot([0,1],[0,1], ls='--', color='#898781')
    for col, c, lab in [('p_b2','#eb6834','B2'), ('p_m','#1baf7a','model')]:
        rep, rel, (fp, mp, n) = library_report(te, col)
        ax.plot(mp, fp, marker='o', color=c, label=f"{lab} (ECE {rep['ece']:.3f})")
        print(lab, {k: round(v, 4) for k, v in rep.items()})
    ax.set_xlabel('forecast P(bust)'); ax.set_ylabel('observed frequency'); ax.legend(frameon=False)

## hi_bust (heavy-rain miss / false alarm): ROC-AUC and PR-AUC of P(bust) and of the ▲ rule

In [7]:
if HAVE:
    from sklearn.metrics import roc_auc_score, average_precision_score
    h = te[te.hi_bust.notna()]
    print('hi_bust base rate', round(h.hi_bust.mean(), 4))
    for col in ['p_b2', 'p_m']:
        print(col, 'ROC', round(roc_auc_score(h.hi_bust, h[col]), 4), 'PR', round(average_precision_score(h.hi_bust, h[col]), 4))

## Error heads: pinball loss (q50, q90) by lead

In [8]:
if HAVE:
    from sklearn.metrics import mean_pinball_loss
    display(te.groupby('lead').apply(lambda g: pd.Series(dict(q50=mean_pinball_loss(g.log_err, g.err_q50, alpha=0.5), q90=mean_pinball_loss(g.log_err, g.err_q90, alpha=0.9), cover90=(g.log_err <= g.err_q90).mean()))).round(4))

## Ablations A1–A7 (definitions frozen in `configs/ablations.yaml`)

In [9]:
if HAVE:
    import yaml
    ab = yaml.safe_load(open('../configs/ablations.yaml'))['ablations']
    st = pd.DataFrame(R['primary']).set_index('model')
    rows = [dict(ablation=k, desc=v['desc'], **st.loc[k, ['bss_b0','bss_b2','bss_b2_lo','bss_b2_hi','pr_auc']].to_dict()) for k, v in ab.items() if k in st.index]
    display(pd.DataFrame(rows).round(4))

## Case studies (frozen list, `configs/cases.yaml`)

In [10]:
if HAVE:
    import yaml
    cases = yaml.safe_load(open('../configs/cases.yaml'))['cases']
    allp = pd.read_parquet('../data/processed/eval/loyo_predictions.parquet')
    for c in cases:
        s = allp[allp.init == pd.Timestamp(c['init'])]
        if len(s): print(c['name'], '| split', c['split'], '| Brier model', round(((s.p_m - s.bust)**2).mean(), 4), '| B2', round(((s.p_b2 - s.bust)**2).mean(), 4), '| busts', int(s.bust.sum()))